$$
   D_{\phi, \sigma}(T) = \sum_{i} ( \bar{F} (\phi, \sigma; T) - F_{i}(\phi, \sigma ;T))^2
$$

Then:

$$
\mathcal{A} (T) := \int_{t_0}^T D(t) \cdot dt
$$

We now ask the question on whether the LP's on CFMM's with pro-rate fee allocation are indirectly demanding a hedge for the proposed risk.


Theoretically it has been shown that under certain conditions sophisticated LP's crowd out passive liquidity providers on CFMM's

Empirically this has been validated showing how sophisticated LP's concentrate on high TVL pools and indeed concentrate revenue

Metrics that aim to isolate this risk from the inherent opportunity cost faced by LPs' have been proposed


This approach bundles all this and proposes the cross-sectional variance of fee capture efficiency as a hedgeable state variable against sophisticated LPs crowding out passive LPs.


The deep question is whether there is an implicit demand for such hedge by LP's. We ask:

Does LP expected return decrease when expected dispersion increases?
$\frac{\partial}{\partial D_t} \big ( \mathbb{E} (R)_{LP}\big)$?


This is isomorphic to asking does the pro-rate fee allocation induce a microstructure premium?

If so we can start building instruments for hedging such risk.

## Building the exercise

As suggested by .. we want to isolate the fee adjusted expected returns (APR). This excludes from $\mathbb{E}(R_{LP})$ the LVR, inventory P&L.

Thus the aggregate LP expected fee adjusted returns are defined as:

$$
	\mathbb{E}_{t+1} (R_{LP}) = \frac{\text{fees}_{t+1}}{\text{TVL}_t}
$$

We use the same regression controls as [REF](./) This is:

$$
    R_{LP, t+1} = \beta_0 + \beta_1 D_t + \Theta \cdot \underbrace{(\sigma_t, \text{volume}_t, \ln(\text{TVL}_t))}_{\text{controls}} + \varepsilon \\

    H_0: \beta_1 < 0
$$

This aggregate test -> D predicts next-period LP returns negatively → priced risk.

We choose a daily frequency because of ...




We choose Unichain USDC/DAI 0.01 pool because of  ...

And consider the time span ...

Since we need cross-sectional variance means we  need position-level data for all the days on the time span

> To optimize resources.

Cleaning position entry/exit

Filtering zero-liquidity days

Handling survivorship bias


Thus we want to consider the pool regime time span that better fits. We think about the largest time span where volume was the largest

- Chart of $D_t$ over time

- Chart of LP returns

- Scatter plot: $D_t$ vs $R_{t+1}$

- Regression table (simple OLS with HAC errors)

In [ ]:
import sys
sys.path.insert(0, '..')

from data.DataHandler import PoolEntryData, delta, tvlUSD, variance, priceUSD, div, lagged
from data.Econometrics import LiquidityStateModel, beta, rho, state, result

pool_id = "0x395f91b34aa34a477ce3bc6505639a821b286a62b1a164fc1887fa3a5ef713a5"
pool = PoolEntryData(pool_id)
poolData = pool(90)

endog = div(delta(tvlUSD(poolData)), lagged(tvlUSD(poolData)))
exog = variance(div(delta(priceUSD(poolData)), lagged(priceUSD(poolData))), None)

ls = LiquidityStateModel()(endog=endog, exog=exog, window=30)

print(f"β₁ = {beta(ls):.6f}  (expect < 0)")
print(f"γ  = {rho(ls):.6f}  (expect 0 < γ < 1)")
print(f"State length: {len(state(ls))}")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

# Classic monochrome template
classic = go.layout.Template()
classic.layout = go.Layout(
    font=dict(family="Courier New, monospace", size=12, color="#1a1a1a"),
    paper_bgcolor="#fafaf5",
    plot_bgcolor="#fafaf5",
    title=dict(font=dict(size=16, family="Courier New, monospace")),
    xaxis=dict(
        showgrid=True, gridcolor="#cccccc", gridwidth=0.5,
        linecolor="#1a1a1a", linewidth=1, mirror=True,
        zeroline=True, zerolinecolor="#999999", zerolinewidth=0.8
    ),
    yaxis=dict(
        showgrid=True, gridcolor="#cccccc", gridwidth=0.5,
        linecolor="#1a1a1a", linewidth=1, mirror=True,
        zeroline=True, zerolinecolor="#999999", zerolinewidth=0.8
    ),
    colorway=["#1a1a1a", "#666666", "#999999", "#bbbbbb"],
)
pio.templates["classic"] = classic
pio.templates.default = "classic"

fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
    subplot_titles=("ΔL_t  (TVL change)", "σ²_t  (price variance)",
                    f"e_t  (structural residual,  γ = {rho(ls):.4f})")
)

fig.add_trace(go.Scatter(
    x=endog.index, y=endog.values, mode="lines",
    line=dict(color="#1a1a1a", width=1), showlegend=False
), row=1, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="#999999", line_width=0.5, row=1, col=1)

fig.add_trace(go.Scatter(
    x=exog.index, y=exog.values, mode="lines",
    line=dict(color="#1a1a1a", width=1), showlegend=False
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=state(ls).index, y=state(ls).values, mode="lines",
    line=dict(color="#1a1a1a", width=1), showlegend=False
), row=3, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="#999999", line_width=0.5, row=3, col=1)

fig.update_layout(height=750, margin=dict(t=40, b=30))
fig.show()

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=exog.values, y=endog.values, mode="markers",
    marker=dict(color="#1a1a1a", size=5, opacity=0.5,
                line=dict(color="#1a1a1a", width=0.5)),
    showlegend=False
))

fig.update_layout(
    title=f"Volatility vs Liquidity Changes  (β₁ = {beta(ls):.4f})",
    xaxis_title="σ²_t  (price variance)",
    yaxis_title="ΔL_t  (TVL change)",
    height=500
)

fig.show()

print(result(ls).summary())